In [65]:
import os
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import transforms

from sklearn.metrics import accuracy_score

from tqdm import tqdm

In [66]:
torch.backends.cudnn.benchmark = True

In [67]:
TRAIN_DIR = "data/Processed_Images/Processed_Train/"
VALID_DIR = "data/Processed_Images/Processed_valid/"
TEST_DIR = "data/Processed_Images/Processed_test/"

TRAIN_CSV = "data/Processed_Images/Processed_Train/Processed_train_annotations.csv"
VALID_CSV = "data/Processed_Images/Processed_valid/Processed_valid_annotations.csv"
TEST_CSV = "data/Processed_Images/Processed_test/Processed_test_annotations.csv"

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [68]:
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-4
TRAIN_RATIO = 0.8
RANDOM_SEED = 42
WEIGHT_DECAY = 1e-4

In [69]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(DEVICE)

cuda


In [70]:
train_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])

In [71]:
valid_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),

])

In [72]:
class SnakeDataset(Dataset):

    def __init__(self, csv_path, image_folder, transform=None):
        """
        Args:
            csv_path (str): Path to processed_annotations.csv
            image_folder (str): Folder containing cropped images
            transform (callable): torchvision transforms
        """

        self.annotations = pd.read_csv(csv_path)
        self.image_folder = image_folder
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):

        row = self.annotations.iloc[index]

        image_path = os.path.join(
            self.image_folder,
            row["filename"]
        )

        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found:\n{image_path}")

        with Image.open(image_path) as image:
            image = image.convert("RGB")

            if self.transform is not None:
                image = self.transform(image)

        label = int(row["Label"])
        
        return image, label

    @property
    def classes(self):
        return sorted(
            self.annotations["Class"].unique().tolist()
        )

    @property
    def num_classes(self):
        return self.annotations["Label"].nunique()

In [73]:
train_dataset = SnakeDataset(
    csv_path=TRAIN_CSV,
    image_folder=TRAIN_DIR,
    transform=train_transform
)

valid_dataset = SnakeDataset(
    csv_path=VALID_CSV,
    image_folder=VALID_DIR,
    transform=valid_transform
)

test_dataset = SnakeDataset(
    csv_path=TEST_CSV,
    image_folder=TEST_DIR,
    transform=valid_transform
)

In [74]:
print(f"Train Images      : {len(train_dataset)}")
print(f"Validation Images : {len(valid_dataset)}")
print(f"Test Images       : {len(test_dataset)}")

print(f"Classes           : {train_dataset.num_classes}")

Train Images      : 6108
Validation Images : 572
Test Images       : 290
Classes           : 15


In [75]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

In [76]:
images, labels = next(iter(train_loader))

print(images.shape)

torch.Size([16, 3, 224, 224])


In [77]:
print(len(train_dataset))
print(len(valid_dataset))

6108
572


In [78]:
import platform
print(platform.processor())

Intel64 Family 6 Model 141 Stepping 1, GenuineIntel


In [79]:
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1

model = convnext_tiny(weights=weights)

num_features = model.classifier[2].in_features

model.classifier[2] = nn.Linear(
    num_features,
    15
)

model.to(DEVICE)

print("=" * 50)
print("Model Loaded Successfully")
print("=" * 50)
print(f"Architecture : ConvNeXt Tiny")
print(f"Classes      : {15}")
print(f"Device       : {DEVICE}")

Model Loaded Successfully
Architecture : ConvNeXt Tiny
Classes      : 15
Device       : cuda


In [80]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

In [81]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [82]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [83]:
scaler = torch.amp.GradScaler("cuda")

In [84]:
print("=" * 50)
print("Training Configuration")
print("=" * 50)

print(f"Model         : ConvNeXt Tiny")
print(f"Classes       : {15}")
print(f"Image Size    : {IMAGE_SIZE}")
print(f"Batch Size    : {BATCH_SIZE}")
print(f"Epochs        : {EPOCHS}")
print(f"Learning Rate : {LEARNING_RATE}")
print(f"Device        : {DEVICE}")
print(f"Optimizer     : {optimizer.__class__.__name__}")
print(f"Scheduler     : {scheduler.__class__.__name__}")
print(f"Loss          : {criterion.__class__.__name__}")
print("=" * 50)

Training Configuration
Model         : ConvNeXt Tiny
Classes       : 15
Image Size    : 224
Batch Size    : 16
Epochs        : 20
Learning Rate : 0.0001
Device        : cuda
Optimizer     : AdamW
Scheduler     : CosineAnnealingLR
Loss          : CrossEntropyLoss


In [85]:
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

LAST_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pth")
BEST_MODEL = os.path.join(CHECKPOINT_DIR, "best_model.pth")

In [86]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        loader,
        total=len(loader),
        desc="Training",
        dynamic_ncols=True,
        leave=False
    )

    for batch_idx, (images, labels) in enumerate(progress_bar):

        batch_start = time.time()

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=False
        ):

            outputs = model(images)
            loss = criterion(outputs, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.detach().item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        avg_loss = running_loss / (batch_idx + 1)
        avg_acc = 100.0 * correct / total

        batch_time = time.time() - batch_start

        gpu_memory = (
            torch.cuda.memory_allocated(device) / 1024**3
            if torch.cuda.is_available()
            else 0
        )

        current_lr = optimizer.param_groups[0]["lr"]

        progress_bar.set_postfix(
           Loss=f"{avg_loss:.4f}",
            Acc=f"{avg_acc:.2f}%",
            LR=f"{current_lr:.2e}",
            GPU=f"{gpu_memory:.2f}GB",
            Batch=f"{batch_time:.2f}s"
        )

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

In [87]:
@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        loader,
        total=len(loader),
        desc="Validation",
        dynamic_ncols=True,
        leave=False
    )

    for batch_idx, (images, labels) in enumerate(progress_bar):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=device.type == "cuda"
        ):

            outputs = model(images)
            if batch_idx < 3:
                print(f"\nBatch {batch_idx}")
                # print(f"Output Min : {outputs.min().item():.4f}")
                # print(f"Output Max : {outputs.max().item():.4f}")
                print(f"Output NaN : {torch.isnan(outputs).any().item()}")
            
            loss = criterion(outputs, labels)
            
            if batch_idx < 3:
                print(f"Loss : {loss.item()}")
            
            print("Loss :", loss.item())

        running_loss += loss.detach().item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        avg_loss = running_loss / (batch_idx + 1)
        avg_acc = 100.0 * correct / total

        progress_bar.set_postfix(
            Loss=f"{avg_loss:.4f}",
            Acc=f"{avg_acc:.2f}%"
        )   

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

In [88]:
# ==========================
# Resume Training (Optional)
# ==========================

start_epoch = 0
best_accuracy = 0.0

history = {
    "train_loss": [],
    "train_acc": [],
    "valid_loss": [],
    "valid_acc": []
}

if os.path.exists(LAST_CHECKPOINT):

    print("=" * 60)
    print("Resuming Training")
    print("=" * 60)

    checkpoint = torch.load(
        LAST_CHECKPOINT,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    scaler.load_state_dict(
        checkpoint["scaler_state_dict"]
    )

    start_epoch = checkpoint["epoch"] + 1
    best_accuracy = checkpoint["best_accuracy"]

    print(f"Resuming from Epoch : {start_epoch}")
    print(f"Best Accuracy       : {best_accuracy:.2f}%")
    print("=" * 60)

else:

    print("=" * 60)
    print("No checkpoint found.")
    print("Training will start from scratch.")
    print("=" * 60)

Resuming Training


Resuming from Epoch : 2
Best Accuracy       : 73.25%


In [89]:
train_df = pd.read_csv(TRAIN_CSV)

train_df = train_df.dropna().reset_index(drop=True)

train_df["Label"] = train_df["Label"].astype(int)

train_df.to_csv(TRAIN_CSV, index=False)

print("CSV repaired.")
print("Rows:", len(train_df))

CSV repaired.
Rows: 6108


In [90]:
train_df = pd.read_csv(TRAIN_CSV)

print(train_df.isna().sum())

filename    0
width       0
height      0
class       0
xmin        0
ymin        0
xmax        0
ymax        0
Label       0
dtype: int64


In [91]:
missing = []

for file in train_df["filename"]:

    if not os.path.exists(os.path.join(TRAIN_DIR, file)):
        missing.append(file)

print("Missing:", len(missing))

Missing: 0


In [92]:
import time

images, labels = next(iter(train_loader))

print("Batch Loaded")

images = images.to(DEVICE)
labels = labels.to(DEVICE)

print("Moved to GPU")

model.train()

start = time.time()
outputs = model(images)
print("Forward :", time.time() - start)

start = time.time()
loss = criterion(outputs, labels)
print("Loss :", time.time() - start)

start = time.time()
optimizer.zero_grad(set_to_none=True)
loss.backward()
print("Backward :", time.time() - start)

start = time.time()
optimizer.step()
print("Optimizer :", time.time() - start)

print("Done")

Batch Loaded
Moved to GPU
Forward : 0.02401113510131836
Loss : 0.002009153366088867
Backward : 0.10552048683166504
Optimizer : 0.563147783279419
Done


In [ ]:
import time 

for epoch in range(start_epoch, EPOCHS):

    print("\n" + "=" * 60)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("=" * 60)

    # ==========================
    # Training
    # ==========================

    train_loss, train_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE
    )

    # ==========================
    # Validation
    # ==========================

    valid_loss, valid_acc = validate(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=DEVICE
    )

    # ==========================
    # Save History
    # ==========================

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["valid_loss"].append(valid_loss)
    history["valid_acc"].append(valid_acc)

    # ==========================
    # Update Learning Rate
    # ==========================

    scheduler.step()

    # ==========================
    # Epoch Summary
    # ==========================

    print(f"\nTrain Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.2f}%")
    print(f"Valid Loss : {valid_loss:.4f}")
    print(f"Valid Acc  : {valid_acc:.2f}%")

    # ==========================
    # Save Last Checkpoint
    # ==========================

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_accuracy": best_accuracy,
        },
        LAST_CHECKPOINT
    )

    # ==========================
    # Save Best Model
    # ==========================

    if valid_acc > best_accuracy:

        best_accuracy = valid_acc

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "best_accuracy": best_accuracy,
            },
            BEST_MODEL
        )

        print(f"✅ Best model saved ({best_accuracy:.2f}%)")

    else:

        print(f"No improvement (Best: {best_accuracy:.2f}%)")


Epoch 3/20


Training:   0%|          | 0/382 [00:00<?, ?it/s]

Training:  34%|███▍      | 130/382 [01:18<02:30,  1.68it/s, Acc=96.25%, Batch=0.48s, GPU=0.99GB, LR=9.76e-05, Loss=0.6978]

In [ ]:
import torch

ckpt = torch.load(LAST_CHECKPOINT, map_location="cpu")

print("Saved Epoch :", ckpt["epoch"] + 1)
print("Best Accuracy :", ckpt["best_accuracy"])

Saved Epoch : 2
Best Accuracy : 73.25174825174825
